# Sprint 4 — O GPT passo a passo: do bloco à geração de texto

**O que a Sprint 3 entregou:** uma `MultiHeadAttention` que transforma `(B, T, emb_dim)` em
`(B, T, emb_dim)`, com máscara causal. Cada token sai com um vetor que leva em conta os anteriores.

**O que falta para um modelo de linguagem:** as peças em volta da atenção (LayerNorm,
FeedForward, conexões de atalho), o empilhamento em vários blocos, a projeção no vocabulário e o
laço que transforma previsões em texto.

Este notebook percorre o capítulo 4 **na ordem do livro** (figura 4.3), mas usando o código de
`src/`, e confere cada etapa contra o número impresso no livro. A ordem:

1. o esqueleto do modelo (seção 4.1)
2. LayerNorm (4.2)
3. GELU e FeedForward (4.3)
4. conexões de atalho (4.4)
5. o Transformer Block (4.5)
6. o GPT completo (4.6)
7. geração de texto (4.7) e estratégias de decodificação (5.3)

A versão detalhada de cada etapa está em `docs/sprint04/`; os experimentos, em `experimentos/sprint04/`.

In [1]:
import sys
from dataclasses import replace
from pathlib import Path

# Convenção do projeto: `jupyter notebook` é aberto na raiz do repositório (ver README),
# então o kernel deste notebook roda com cwd = notebooks/sprint04/. Dois níveis acima é a raiz.
PROJECT_ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn as nn

from src.generation import generate, next_token_probabilities, select_next_token, text_to_token_ids, token_ids_to_text
from src.model import BASE_CONFIG, GPT_CONFIG_124M, GPTModel
from src.tokenizer import BPETokenizer
from src.transformer import GELU, FeedForward, LayerNorm, ShortcutDemoNetwork, TransformerBlock, mean_weight_gradients
from tests import book_reference as livro  # valores impressos no livro, com a página

torch.set_printoptions(sci_mode=False, precision=4)
bpe = BPETokenizer()

print("Configuração do livro :", GPT_CONFIG_124M)
print("Configuração do projeto:", BASE_CONFIG)

Configuração do livro : GPTConfig(vocab_size=50257, context_length=1024, emb_dim=768, n_heads=12, n_layers=12, drop_rate=0.1, qkv_bias=False, ff_hidden_dim=None, drop_rate_emb=None, drop_rate_attn=None, drop_rate_shortcut=None, tie_weights=False)
Configuração do projeto: GPTConfig(vocab_size=50257, context_length=128, emb_dim=256, n_heads=4, n_layers=4, drop_rate=0.1, qkv_bias=False, ff_hidden_dim=None, drop_rate_emb=None, drop_rate_attn=None, drop_rate_shortcut=None, tie_weights=False)


## 1. O esqueleto (seção 4.1)

O livro começa pelo fluxo de dados, com peças falsas no lugar do bloco e da LayerNorm
(`DummyGPTModel`). No projeto, o esqueleto não é outra classe: é o próprio `GPTModel` com
**zero blocos** e a norma final trocada pela identidade. Como blocos inexistentes não sorteiam
pesos, a seed 123 produz os mesmos números do livro.

In [2]:
lote = torch.tensor([bpe.encode(texto) for texto in livro.BATCH_TEXTS])
print("Token IDs:\n", lote)

torch.manual_seed(livro.BOOK_SEED)
esqueleto = GPTModel(replace(GPT_CONFIG_124M, n_layers=0))
esqueleto.final_norm = nn.Identity()

with torch.no_grad():
    logits = esqueleto(lote)   # em train(), como no livro: o dropout participa

print("\nshape dos logits:", tuple(logits.shape), "-> (lote, tokens, vocabulário)")
print("texto 0, posição 0 :", logits[0, 0, :3], "...", logits[0, 0, -3:])
print("livro (p. 99)      :", livro.DUMMY_LOGITS[0][0])

Token IDs:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])



shape dos logits: (2, 4, 50257) -> (lote, tokens, vocabulário)
texto 0, posição 0 : tensor([-1.2034,  0.3201, -0.7130]) ... tensor([-1.5548, -0.2390, -0.4667])
livro (p. 99)      : ([-1.2034, 0.3201, -0.713], [-1.5548, -0.239, -0.4667])


Já se vê o contrato do modelo: **um vetor de 50 257 logits por token de entrada**. A linha `t`
será a previsão do token `t + 1`. Falta o que fica entre os embeddings e a saída.

## 2. LayerNorm (seção 4.2)

A LayerNorm leva cada vetor a média 0 e variância 1, na última dimensão. O exemplo do livro: uma
camada `Linear(5, 6) + ReLU` sobre duas entradas aleatórias.

In [3]:
torch.manual_seed(livro.BOOK_SEED)
entrada = torch.randn(2, 5)
camada = nn.Sequential(nn.Linear(5, 6), nn.ReLU())

with torch.no_grad():
    saida = camada(entrada)
print("saída da camada:\n", saida)
print("média por linha    :", saida.mean(dim=-1), "  livro:", livro.LAYER_OUTPUT_MEAN)
print("variância por linha:", saida.var(dim=-1), "  livro:", livro.LAYER_OUTPUT_VAR)

with torch.no_grad():
    normalizado = LayerNorm(emb_dim=5)(entrada)   # src/transformer/layer_norm.py
print("\nLayerNorm do projeto -> média:", normalizado.mean(dim=-1), " variância:", normalizado.var(dim=-1, unbiased=False))

saída da camada:
 tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]])
média por linha    : tensor([0.1324, 0.2170])   livro: [0.1324, 0.217]
variância por linha: tensor([0.0231, 0.0398])   livro: [0.0231, 0.0398]

LayerNorm do projeto -> média: tensor([-0.0000,  0.0000])  variância: tensor([1.0000, 1.0000])


Média 0 e variância 1 em cada linha, independentemente das outras linhas: é por isso que a
LayerNorm funciona com lote de 1 na geração, ao contrário da BatchNorm. `scale` e `shift`
começam em 1 e 0 e são treináveis — o modelo pode desfazer a normalização, canal a canal.

## 3. GELU e FeedForward (seção 4.3)

A FeedForward expande cada token para 4x a dimensão, aplica a GELU e contrai de volta. A GELU
difere da ReLU nos negativos: não os zera.

In [4]:
x = torch.tensor([-3.0, -1.0, -0.75, -0.5, 0.0, 0.5, 1.0, 3.0])
with torch.no_grad():
    print("x       :", x)
    print("GELU(x) :", GELU()(x))
    print("ReLU(x) :", nn.ReLU()(x))

torch.manual_seed(livro.BOOK_SEED)
ffn = FeedForward(GPT_CONFIG_124M.emb_dim)   # 768 -> 3072 -> 768
h = torch.rand(2, 3, 768)
print("\nentrada da FeedForward :", tuple(h.shape))
with torch.no_grad():
    for etapa in ffn.layers:
        h = etapa(h)
        print(f"depois de {type(etapa).__name__:<7}:", tuple(h.shape))

x       : tensor([-3.0000, -1.0000, -0.7500, -0.5000,  0.0000,  0.5000,  1.0000,  3.0000])
GELU(x) : tensor([-0.0036, -0.1588, -0.1700, -0.1543,  0.0000,  0.3457,  0.8412,  2.9964])
ReLU(x) : tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 1.0000, 3.0000])

entrada da FeedForward : (2, 3, 768)
depois de Linear : (2, 3, 3072)
depois de GELU   : (2, 3, 3072)
depois de Linear : (2, 3, 768)


O shape volta ao de entrada (figura 4.9). Essa igualdade não é estética: é a condição para a
saída ser **somada** à entrada na conexão de atalho.

## 4. Conexões de atalho (seção 4.4)

A rede de demonstração do livro: cinco camadas Linear + GELU, com e sem `x = x + camada(x)`.
O que se compara é o gradiente médio que chega a cada camada.

In [5]:
entrada = torch.tensor(livro.SHORTCUT_INPUT)
for usa_atalho, esperado in ((False, livro.GRADIENTS_WITHOUT_SHORTCUT), (True, livro.GRADIENTS_WITH_SHORTCUT)):
    torch.manual_seed(livro.BOOK_SEED)
    rede = ShortcutDemoNetwork(livro.SHORTCUT_LAYER_SIZES, use_shortcut=usa_atalho)
    obtido = mean_weight_gradients(rede, entrada)
    print("com atalho" if usa_atalho else "sem atalho")
    for camada, (o, e) in enumerate(zip(obtido, esperado)):
        print(f"  layers.{camada}: {o:.8f}   livro: {e:.8f}")

sem atalho
  layers.0: 0.00020174   livro: 0.00020174
  layers.1: 0.00012011   livro: 0.00012011
  layers.2: 0.00071520   livro: 0.00071520
  layers.3: 0.00139887   livro: 0.00139887
  layers.4: 0.00504965   livro: 0.00504965
com atalho
  layers.0: 0.22169794   livro: 0.22169793
  layers.1: 0.20694102   livro: 0.20694105
  layers.2: 0.32896990   livro: 0.32896996
  layers.3: 0.26657322   livro: 0.26657325
  layers.4: 1.32585418   livro: 1.32585418


Sem atalho, o gradiente da primeira camada é 25 vezes menor que o da última (*vanishing
gradient*); com atalho, ele se mantém da ordem de 0,2. A derivada de `x + f(x)` é `1 + f'(x)`:
o "1" deixa o gradiente atravessar a camada sem ser multiplicado.

## 5. O Transformer Block (seção 4.5)

As quatro peças em volta da atenção da Sprint 3, em duas linhas
(`src/transformer/block.py`):

```
x = x + Dropout(Atenção(LayerNorm₁(x)))
x = x + Dropout(FeedForward(LayerNorm₂(x)))
```

In [6]:
torch.manual_seed(livro.BOOK_SEED)
x = torch.rand(2, 4, 768)
bloco = TransformerBlock(emb_dim=768, context_length=1024, n_heads=12, drop_rate=0.1)

with torch.no_grad():
    print("entrada:", tuple(x.shape), " saída:", tuple(bloco(x).shape))

atencao = sum(p.numel() for p in bloco.att.parameters())
feed_forward = sum(p.numel() for p in bloco.ff.parameters())
print(f"\nexercício 4.1 -> atenção: {atencao:,}   FeedForward: {feed_forward:,}   razão: {feed_forward / atencao:.2f}")

# O atalho é o caminho padrão: zerando a última projeção dos dois ramos, o bloco vira a identidade.
bloco.eval()
with torch.no_grad():
    for linear in (bloco.att.out_proj, bloco.ff.layers[2]):
        linear.weight.zero_()
        linear.bias.zero_()
    print("com os ramos zerados, saída == entrada:", torch.equal(bloco(x), x))

entrada: (2, 4, 768)  saída: (2, 4, 768)

exercício 4.1 -> atenção: 2,360,064   FeedForward: 4,722,432   razão: 2.00
com os ramos zerados, saída == entrada: True


O bloco devolve o shape que recebe, e por isso pode ser empilhado. A FeedForward tem o dobro
dos parâmetros da atenção. E cada subcamada é uma **correção** somada ao fluxo residual, não
uma substituição — com os ramos zerados, sobra exatamente a entrada.

## 6. O GPT completo (seção 4.6)

Embeddings (Sprint 2) → 12 blocos → LayerNorm final → cabeça de saída. Com a seed 123 e a mesma
ordem de criação dos pesos, os logits devem ser os da página 120 do livro.

In [7]:
torch.manual_seed(livro.BOOK_SEED)
gpt = GPTModel(GPT_CONFIG_124M)
with torch.no_grad():
    logits = gpt(lote)

print("texto 0, posição 0:", logits[0, 0, :3], "...", logits[0, 0, -3:])
print("livro (p. 120)    :", livro.GPT_LOGITS[0][0])
print("texto 1, posição 3:", logits[1, 3, :3], "...", logits[1, 3, -3:])
print("livro (p. 120)    :", livro.GPT_LOGITS[1][3])

total = gpt.parameter_count()
cabeca = gpt.parameter_breakdown()["out_head"]
print(f"\nparâmetros: {total:,}  (livro: {livro.GPT_124M_PARAMS:,})")
print(f"sem a cabeça (weight tying): {total - cabeca:,}  (livro: {livro.GPT_124M_PARAMS_TIED:,})")
print(f"tamanho em float32: {gpt.size_in_megabytes():.2f} MB  (livro: {livro.GPT_124M_MEGABYTES} MB)")
for componente, n in gpt.parameter_breakdown().items():
    print(f"  {componente:<22}{n:>13,}  {100 * n / total:5.1f}%")

texto 0, posição 0: tensor([ 0.3613,  0.4222, -0.0711]) ... tensor([ 0.3483,  0.4661, -0.2838])
livro (p. 120)    : ([0.3613, 0.4222, -0.0711], [0.3483, 0.4661, -0.2838])
texto 1, posição 3: tensor([-0.1565,  0.3926,  0.3288]) ... tensor([ 1.2630, -0.1858,  0.0388])
livro (p. 120)    : ([-0.1565, 0.3926, 0.3288], [1.263, -0.1858, 0.0388])

parâmetros: 163,009,536  (livro: 163,009,536)
sem a cabeça (weight tying): 124,412,160  (livro: 124,412,160)
tamanho em float32: 621.83 MB  (livro: 621.83 MB)
  token_embedding          38,597,376   23.7%
  positional_embedding        786,432    0.5%
  attention                28,320,768   17.4%
  feed_forward             56,669,184   34.8%
  block_norms                  36,864    0.0%
  final_norm                    1,536    0.0%
  out_head                 38,597,376   23.7%


Os logits batem com o livro na quarta casa: embeddings, 12 blocos, norma final e cabeça fazem
a mesma conta, com os mesmos pesos. A divisão dos parâmetros explica os "163 milhões em vez de
124": a cabeça de saída tem 38,6 milhões, que o GPT-2 original não conta porque reaproveita a
tabela de tokens (*weight tying*).

### O modelo do projeto, com dados de verdade

A `BASE_CONFIG` é a configuração das Sprints 2 e 3 com 4 blocos. Um lote do The Verdict pelo
`DataLoader` da Sprint 2 já permite calcular a perda do próximo token.

In [8]:
import math
import torch.nn.functional as F
from src.corpus import load_verdict
from src.dataset import create_dataloader

loader = create_dataloader(load_verdict(), bpe, context_length=128, stride=128, batch_size=8, shuffle=False)
entrada, alvo = next(iter(loader))

torch.manual_seed(42)
modelo = GPTModel(BASE_CONFIG).eval()
with torch.no_grad():
    logits = modelo(entrada)

perda = F.cross_entropy(logits.flatten(0, 1), alvo.flatten())
print("Token IDs:", tuple(entrada.shape), "-> logits:", tuple(logits.shape))
print(f"parâmetros: {modelo.parameter_count():,}")
print(f"perda inicial: {perda.item():.3f}   ln(V) = {math.log(BASE_CONFIG.vocab_size):.3f} (chute uniforme)")

Token IDs: (8, 128) -> logits: (8, 128, 50257)
parâmetros: 28,920,832
perda inicial: 10.976   ln(V) = 10.825 (chute uniforme)


Sem treino, a perda é praticamente a de quem distribui a probabilidade igualmente entre os
50 257 tokens: o modelo ainda não sabe nada. É o ponto de partida da Sprint 5.

## 7. Geração de texto (seção 4.7)

Um passo de geração: passada pelo modelo, **só a última linha** de logits, softmax, escolha.
Depois, o token escolhido é anexado ao contexto e o passo se repete — é a geração
autoregressiva, implementada em `generate` (`src/generation/autoregressive.py`).

In [9]:
ids = text_to_token_ids(livro.GENERATION_PROMPT, bpe)
gpt.eval()
with torch.no_grad():
    ultimo = gpt(ids)[:, -1, :]                 # (1, T, V) -> (1, V)
probabilidades = torch.softmax(ultimo, dim=-1)
escolhido = select_next_token(ultimo)           # temperatura 0: argmax

print("prompt:", ids.tolist(), "->", repr(livro.GENERATION_PROMPT))
print("soma das probabilidades:", probabilidades.sum().item())
print("argmax dos logits == argmax das probabilidades:", ultimo.argmax().item() == probabilidades.argmax().item())
print("próximo token:", escolhido.item(), repr(bpe.decode([escolhido.item()])))

saida = generate(gpt, ids, max_new_tokens=livro.GENERATION_NEW_TOKENS)
print("\ngerado:", saida[0].tolist())
print("livro :", livro.GENERATION_OUTPUT_IDS)
print("texto :", repr(token_ids_to_text(saida, bpe)))

prompt: [[15496, 11, 314, 716]] -> 'Hello, I am'
soma das probabilidades: 0.9999997615814209
argmax dos logits == argmax das probabilidades: True
próximo token: 27018 ' Feature'

gerado: [15496, 11, 314, 716, 27018, 24086, 47843, 30961, 42348, 7267]
livro : [15496, 11, 314, 716, 27018, 24086, 47843, 30961, 42348, 7267]
texto : 'Hello, I am Featureiman Byeswickattribute argue'


O texto é incoerente pelo motivo que o livro dá — os pesos são os sorteados pela seed —, mas é
o **mesmo** incoerente do livro, token a token.

## 8. Estratégias de decodificação (seção 5.3)

O livro usa um vocabulário de 9 palavras com logits fixos para isolar o que a temperatura e o
top-k fazem. As mesmas funções do projeto, sobre esses logits:

In [10]:
logits_brinquedo = torch.tensor(livro.TOY_LOGITS)
print(f"{'palavra':<9}" + "".join(f"{'T=' + format(t, 'g'):>9}" for t in livro.TOY_TEMPERATURES) + f"{'top-3':>9}")
tabelas = [next_token_probabilities(logits_brinquedo, temperature=t) for t in livro.TOY_TEMPERATURES]
top3 = next_token_probabilities(logits_brinquedo, top_k=3)
for i, palavra in enumerate(livro.TOY_VOCAB):
    print(f"{palavra:<9}" + "".join(f"{p[i].item():>9.4f}" for p in tabelas) + f"{top3[i].item():>9.4f}")

gerador = torch.Generator().manual_seed(livro.BOOK_SEED)
sorteios = [select_next_token(logits_brinquedo.unsqueeze(0), temperature=1.0, generator=gerador).item() for _ in range(1000)]
contagem = torch.bincount(torch.tensor(sorteios), minlength=9)
print("\n1000 sorteios com T = 1:", {p: c for p, c in zip(livro.TOY_VOCAB, contagem.tolist()) if c})
print("livro (p. 153-154)     :", {p: c for p, c in livro.TOY_SAMPLE_COUNTS.items() if c})

palavra        T=1    T=0.1      T=5    top-3
closer      0.0609   0.0000   0.1546   0.0615
every       0.0016   0.0000   0.0750   0.0000
effort      0.0001   0.0000   0.0429   0.0000
forward     0.5721   0.9910   0.2421   0.5775
inches      0.0034   0.0000   0.0869   0.0000
moves       0.0001   0.0000   0.0454   0.0000
pizza       0.0001   0.0000   0.0430   0.0000
toward      0.3576   0.0090   0.2203   0.3610
you         0.0040   0.0000   0.0898   0.0000



1000 sorteios com T = 1: {'closer': 73, 'forward': 582, 'inches': 2, 'toward': 343}
livro (p. 153-154)     : {'closer': 73, 'forward': 582, 'inches': 2, 'toward': 343}


A temperatura muda a **forma** da distribuição sem mudar a ordem (`forward` é sempre o
primeiro): 0,1 a aproxima da escolha gulosa, 5 a aproxima da uniforme. O top-k zera tudo fora
dos k primeiros, com o mesmo truque de −∞ antes do softmax da máscara causal da Sprint 3.

## O que a Sprint 5 recebe

- Um `GPTModel` configurável que transforma `(B, T)` de Token IDs em `(B, T, V)` de logits, com
  os alvos do `DataLoader` da Sprint 2 casando posição a posição — a perda já pode ser calculada.
- Uma perda inicial de referência: ~11 nats, a de um chute uniforme.
- `generate` para acompanhar o texto produzido ao longo do treino.

O que falta é justamente o que faria o texto deixar de ser incoerente: o laço de treino.